# 02 — Tests de estacionariedad (HICP Eurozona)

Determinacion formal del orden de integracion de la serie HICP.

**Tests:** ADF (H0: raiz unitaria) | KPSS (H0: estacionaria) | Phillips-Perron (H0: raiz unitaria)

Aplicados sobre: nivel, diff(1), diff(12), diff(1)+diff(12).

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller, kpss
from arch.unitroot import PhillipsPerron

NOTEBOOK_DIR = Path(r"c:/Users/usuario/OneDrive/Documentos/GitHub/tfg-ipc-mcp/tfg-forecasting/02_eda")
ROOT = NOTEBOOK_DIR.parent
MONOREPO = ROOT.parent
sys.path.insert(0, str(MONOREPO))
from shared.constants import DATE_TRAIN_END
plt.rcParams.update({"figure.figsize": (14, 4), "axes.grid": True, "grid.alpha": 0.3})

In [ ]:
df = pd.read_parquet(ROOT / "data" / "processed" / "hicp_europe_index.parquet")
df["date"] = pd.to_datetime(df["date"])
df = df.set_index("date")
train = df.loc[:DATE_TRAIN_END]
y = train["hicp_index"]
print(f"Train: {y.index.min().date()} -> {y.index.max().date()} ({len(y)} obs)")

## 1. Funciones auxiliares

In [ ]:
def run_adf(series, name=""):
    result = adfuller(series.dropna(), autolag="AIC")
    stat, pval, lags, nobs = result[0], result[1], result[2], result[3]
    cv = result[4]
    reject = pval < 0.05
    return {"test": "ADF", "serie": name, "statistic": round(stat,4), "p_value": round(pval,4),
            "lags": lags, "nobs": nobs, "cv_5%": round(cv["5%"],4),
            "reject_H0": reject, "conclusion": "Estacionaria" if reject else "No estacionaria"}

def run_kpss(series, name="", regression="c"):
    stat, pval, lags, cv = kpss(series.dropna(), regression=regression, nlags="auto")
    reject = pval < 0.05
    return {"test": f"KPSS({regression})", "serie": name, "statistic": round(stat,4), "p_value": round(pval,4),
            "lags": lags, "cv_5%": round(cv["5%"],4),
            "reject_H0": reject, "conclusion": "No estacionaria" if reject else "Estacionaria"}

def run_pp(series, name=""):
    pp = PhillipsPerron(series.dropna())
    reject = pp.pvalue < 0.05
    return {"test": "Phillips-Perron", "serie": name, "statistic": round(pp.stat,4), "p_value": round(pp.pvalue,4),
            "lags": pp.lags, "reject_H0": reject, "conclusion": "Estacionaria" if reject else "No estacionaria"}

def battery(series, name):
    return [run_adf(series, name), run_kpss(series, name, "c"), run_kpss(series, name, "ct"), run_pp(series, name)]

## 2. Tests sobre la serie en nivel

In [ ]:
results = battery(y, "HICP nivel")
pd.DataFrame(results)

## 3. Primera diferencia (d=1)

In [ ]:
y_diff1 = y.diff().dropna()
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(y.index, y, linewidth=1)
axes[0].set_title("Serie en nivel")
axes[1].plot(y_diff1.index, y_diff1, linewidth=1, color="#d62728")
axes[1].set_title("Primera diferencia (d=1)")
axes[1].axhline(0, color="black", linewidth=0.5)
plt.tight_layout(); plt.show()

In [ ]:
results += battery(y_diff1, "HICP diff(1)")
pd.DataFrame(battery(y_diff1, "HICP diff(1)"))

## 4. Diferencia estacional (D=1, lag 12)

In [ ]:
y_diff12 = y.diff(12).dropna()
fig, ax = plt.subplots()
ax.plot(y_diff12.index, y_diff12, linewidth=1, color="#2ca02c")
ax.set_title("Diferencia estacional (lag 12)")
ax.axhline(0, color="black", linewidth=0.5)
plt.tight_layout(); plt.show()

In [ ]:
results += battery(y_diff12, "HICP diff(12)")
pd.DataFrame(battery(y_diff12, "HICP diff(12)"))

## 5. Doble diferencia: d=1 + D=1

In [ ]:
y_diff1_12 = y.diff().diff(12).dropna()
fig, ax = plt.subplots()
ax.plot(y_diff1_12.index, y_diff1_12, linewidth=1, color="#9467bd")
ax.set_title("Doble diferencia: diff(1) + diff(12)")
ax.axhline(0, color="black", linewidth=0.5)
plt.tight_layout(); plt.show()

In [ ]:
results += battery(y_diff1_12, "HICP diff(1)+diff(12)")
pd.DataFrame(battery(y_diff1_12, "HICP diff(1)+diff(12)"))

## 6. Tabla resumen

In [ ]:
summary = pd.DataFrame(results)
cols_show = ["test", "serie", "statistic", "p_value", "conclusion"]
summary[cols_show].style.map(
    lambda v: "background-color: #d4edda" if v == "Estacionaria" else
              "background-color: #f8d7da" if v == "No estacionaria" else "",
    subset=["conclusion"]
)

## 7. Conclusion

**Interpretacion conjunta ADF + KPSS + PP:**

| Transformacion | ADF | KPSS | PP | Decision |
|---|---|---|---|---|
| Nivel | ? | ? | ? | |
| diff(1) | ? | ? | ? | |
| diff(12) | ? | ? | ? | |
| diff(1)+diff(12) | ? | ? | ? | |

*Completar con valores reales tras ejecucion.*